In [1]:
# [Markdown]
# # 🧠 Lab 02: Alpha Grid Search & Production Training
# **Objective:** Dynamically test different Target Horizons and Sequence Contexts to find the highest ROC-AUC.
# Once found, train the final production `Quantformer` and export the artifacts.

import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import joblib
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
from IPython.display import display

# --- 1. PYTORCH DATASET ---
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y, seq_len):
        self.X, self.y, self.seq_len = X, y, seq_len
        
    def __len__(self): 
        return len(self.X) - self.seq_len
        
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx : idx + self.seq_len], dtype=torch.float32), \
               torch.tensor(self.y[idx + self.seq_len - 1], dtype=torch.float32)

# --- 2. QUANTFORMER ARCHITECTURE ---
class Quantformer(nn.Module):
    def __init__(self, num_features, d_model=64, nhead=4, num_layers=2):
        super().__init__()
        self.proj = nn.Linear(num_features, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*2, 
            batch_first=True, dropout=0.2
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.fc = nn.Sequential(
            nn.Linear(d_model, 32), 
            nn.ReLU(), 
            nn.Dropout(0.2), 
            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        return self.fc(self.transformer(self.proj(x))[:, -1, :])

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"⚡ Compute Device: {device}")

⚡ Compute Device: mps


In [4]:
# [Markdown]
# ### 1. Base Data Loading
# We load the pristine tensor from Lab 01. We define the feature splits here so the Grid Search can efficiently route them.

df_master = pd.read_parquet('../data/processed/clean_tensor.parquet')

# Isolate Macro Embeddings
macro_cols = [c for c in df_master.columns if c.startswith('macro_emb_')]

# Isolate Technical & Structural Features
exclude_cols = ['open', 'high', 'low', 'close', 'volume', 'fwd_return_1h', 'target_dir_1h', 'active_session_name', 'markov_regime']
tech_features = [c for c in df_master.columns if c not in exclude_cols + macro_cols]

print(f"✅ Master Tensor Loaded: {len(df_master)} bars")
print(f"📊 Tech/Structural Features: {len(tech_features)}")
print(f"🌍 Macro Embeddings: {len(macro_cols)}")

✅ Master Tensor Loaded: 99987 bars
📊 Tech/Structural Features: 92
🌍 Macro Embeddings: 384


In [5]:
# [Markdown]
# ### 2. The Grid Search Engine
# Iterates over combinations of prediction horizons and historical context windows.

horizons = [2, 4, 8]       # 30m, 1h, 2h forward prediction
seq_lengths = [16, 32, 64] # 4h, 8h, 16h historical context
results = []

print("🚀 Initiating Horizon vs Sequence Grid Search...\n")

for h in horizons:
    # 1. Dynamically build target
    df = df_master.copy()
    df[f'fwd_ret_{h}'] = np.log(df['close'].shift(-h) / df['close'])
    df[f'target_{h}'] = np.where(df[f'fwd_ret_{h}'] > 0, 1, 0)
    df = df.dropna(subset=[f'fwd_ret_{h}'])
    
    # 2. Split (Chronological)
    split_idx = int(len(df) * 0.8)
    train_df, val_df = df.iloc[:split_idx], df.iloc[split_idx:]
    
    # 3. Macro Compression (Fit on Train only)
    pca = PCA(n_components=8, random_state=42)
    macro_train = pca.fit_transform(train_df[macro_cols])
    macro_val = pca.transform(val_df[macro_cols])
    
    # 4. Feature Merging & Scaling
    X_train_raw = np.hstack([train_df[tech_features].values, macro_train])
    X_val_raw = np.hstack([val_df[tech_features].values, macro_val])
    
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_val_scaled = scaler.transform(X_val_raw)
    
    y_train = train_df[f'target_{h}'].values
    y_val = val_df[f'target_{h}'].values
    
    # 5. Class Weighting
    pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()], dtype=torch.float32).to(device)

    # --- INNER LOOP: Sequence Lengths ---
    for seq in seq_lengths:
        print(f"⏳ Training: Horizon={h} bars | Sequence={seq} bars...", end=" ")
        
        train_loader = DataLoader(TimeSeriesDataset(X_train_scaled, y_train, seq), batch_size=256, shuffle=True, drop_last=True)
        val_loader = DataLoader(TimeSeriesDataset(X_val_scaled, y_val, seq), batch_size=256, shuffle=False)
        
        model = Quantformer(num_features=X_train_scaled.shape[1]).to(device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
        
        best_val_loss, best_auc = float('inf'), 0
        patience, no_improve = 3, 0
        
        for epoch in range(15): # Max epochs per combo for speed
            model.train()
            for X_b, y_b in train_loader:
                optimizer.zero_grad()
                loss = criterion(model(X_b.to(device)), y_b.to(device).unsqueeze(1))
                loss.backward()
                optimizer.step()
                
            model.eval()
            val_losses, y_true, y_pred = [], [], []
            with torch.no_grad():
                for X_b, y_b in val_loader:
                    logits = model(X_b.to(device))
                    val_losses.append(criterion(logits, y_b.to(device).unsqueeze(1)).item())
                    y_true.extend(y_b.numpy())
                    y_pred.extend(torch.sigmoid(logits).cpu().numpy())
            
            avg_val_loss = np.mean(val_losses)
            auc = roc_auc_score(y_true, y_pred)
            
            if avg_val_loss < best_val_loss:
                best_val_loss, best_auc = avg_val_loss, auc
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= patience: break
                
        print(f"✅ Val AUC: {best_auc:.4f}")
        results.append({'Horizon': h, 'Sequence': seq, 'Val_Loss': best_val_loss, 'Val_AUC': best_auc})

# Display Matrix
res_df = pd.DataFrame(results).sort_values(by='Val_AUC', ascending=False)
print("\n🏆 Grid Search Results (Sorted by Predictability):")
display(res_df)

🚀 Initiating Horizon vs Sequence Grid Search...



/Users/mishazuy/Desktop/trading_proj/spider_global/venv/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


⏳ Training: Horizon=2 bars | Sequence=16 bars... ✅ Val AUC: 0.7254
⏳ Training: Horizon=2 bars | Sequence=32 bars... ✅ Val AUC: 0.7322
⏳ Training: Horizon=2 bars | Sequence=64 bars... ✅ Val AUC: 0.7250


/Users/mishazuy/Desktop/trading_proj/spider_global/venv/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


⏳ Training: Horizon=4 bars | Sequence=16 bars... ✅ Val AUC: 0.7349
⏳ Training: Horizon=4 bars | Sequence=32 bars... ✅ Val AUC: 0.7281
⏳ Training: Horizon=4 bars | Sequence=64 bars... ✅ Val AUC: 0.7293


/Users/mishazuy/Desktop/trading_proj/spider_global/venv/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


⏳ Training: Horizon=8 bars | Sequence=16 bars... ✅ Val AUC: 0.7302
⏳ Training: Horizon=8 bars | Sequence=32 bars... ✅ Val AUC: 0.7169
⏳ Training: Horizon=8 bars | Sequence=64 bars... ✅ Val AUC: 0.7204

🏆 Grid Search Results (Sorted by Predictability):


,Horizon,Sequence,Val_Loss,Val_AUC
3,4,16,0.681954,0.734949
1,2,32,0.662094,0.732218
6,8,16,0.687994,0.730210
5,4,64,0.683480,0.729282
4,4,32,0.675475,0.728103
0,2,16,0.669076,0.725361
2,2,64,0.669401,0.724981
8,8,64,0.692295,0.720374
7,8,32,0.701901,0.716941


In [9]:
# [Markdown]
# ### 3. Train Final Model & Export Artifacts
# Automatically takes the top-performing configuration from the Grid Search, 
# trains the final model with full patience, and exports the artifacts.
#int(best_config['Horizon'])
#int(best_config['Sequence'])

best_config = res_df.iloc[0]
best_h = 4
best_seq = 32

print(f"🏭 Retraining Final Model with Optimal Params: Horizon={best_h}, Sequence={best_seq}")

# 1. Rebuild Exact Optimal Dataset
df = df_master.copy()
df[f'fwd_ret_{best_h}'] = np.log(df['close'].shift(-best_h) / df['close'])
df['target'] = np.where(df[f'fwd_ret_{best_h}'] > 0, 1, 0)
df = df.dropna(subset=[f'fwd_ret_{best_h}'])

split_idx = int(len(df) * 0.8)
train_df, val_df = df.iloc[:split_idx], df.iloc[split_idx:]

pca = PCA(n_components=8, random_state=42)
macro_train = pca.fit_transform(train_df[macro_cols])
macro_val = pca.transform(val_df[macro_cols])

X_train_raw = np.hstack([train_df[tech_features].values, macro_train])
X_val_raw = np.hstack([val_df[tech_features].values, macro_val])

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_val_scaled = scaler.transform(X_val_raw)

y_train, y_val = train_df['target'].values, val_df['target'].values
pos_weight = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()], dtype=torch.float32).to(device)

# 2. Final Training Run
train_loader = DataLoader(TimeSeriesDataset(X_train_scaled, y_train, best_seq), batch_size=256, shuffle=True, drop_last=True)
val_loader = DataLoader(TimeSeriesDataset(X_val_scaled, y_val, best_seq), batch_size=256, shuffle=False)

final_model = Quantformer(num_features=X_train_scaled.shape[1]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(final_model.parameters(), lr=1e-4, weight_decay=1e-5)

best_loss = float('inf')
patience, no_improve = 5, 0

for epoch in range(50):
    final_model.train()
    for X_b, y_b in train_loader:
        optimizer.zero_grad()
        loss = criterion(final_model(X_b.to(device)), y_b.to(device).unsqueeze(1))
        loss.backward()
        optimizer.step()
        
    final_model.eval()
    val_losses = []
    with torch.no_grad():
        for X_b, y_b in val_loader:
            val_losses.append(criterion(final_model(X_b.to(device)), y_b.to(device).unsqueeze(1)).item())
            
    avg_loss = np.mean(val_losses)
    if avg_loss < best_loss:
        best_loss = avg_loss
        no_improve = 0
        torch.save(final_model.state_dict(), 'temp_best.pth')
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"🛑 Final Model Early Stopping at epoch {epoch+1}")
            break

# 3. Export to Production Path
save_dir = '../data/processed/'
os.makedirs(save_dir, exist_ok=True)

final_model.load_state_dict(torch.load('temp_best.pth'))
torch.save(final_model.state_dict(), os.path.join(save_dir, 'core_quantformer.pth'))
joblib.dump(scaler, os.path.join(save_dir, 'core_scaler.pkl'))
joblib.dump(pca, os.path.join(save_dir, 'core_pca.pkl'))
joblib.dump(tech_features, os.path.join(save_dir, 'core_features.pkl'))
# We also save the optimal config so the live bot knows how to size its arrays
joblib.dump({'horizon': best_h, 'seq_len': best_seq}, os.path.join(save_dir, 'core_config.pkl'))

print(f"✅ SUCCESS! Production artifacts saved to {save_dir}")

🏭 Retraining Final Model with Optimal Params: Horizon=4, Sequence=32


/Users/mishazuy/Desktop/trading_proj/spider_global/venv/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


🛑 Final Model Early Stopping at epoch 12
✅ SUCCESS! Production artifacts saved to ../data/processed/
